In [ ]:
code = 'CSRE_HCLP'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output/'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [ ]:
def CSRE_HCLP(bt, start_time, end_time, orderside, method, sl, om, re_entries):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None
        
        from_candle_close = True if method == 'CC' else False
        
        entry_time = start_dt
        trades_entries = []
        
        trades_entries.append((ce_scrip, pe_scrip, ce_price, pe_price, start_dt))
        ce_data, pe_data = bt._get_straddle_data(start_dt, end_dt, ce_scrip, pe_scrip, seperate=True)
        scommon_dt = sorted(set(ce_data['date_time']) & set(pe_data['date_time']))
        
        _, _, _, _, ce_sl_time, _ = bt.sl_check_by_given_data(ce_data, o=ce_price, sl=sl, orderside=orderside, from_candle_close=from_candle_close)
        _, _, _, _, pe_sl_time, _ = bt.sl_check_by_given_data(pe_data, o=pe_price, sl=sl, orderside=orderside, from_candle_close=from_candle_close)
        
        sl_flag, exit_time = False, ''
        for re_no in range(max_re):
            
            ce_sl_time = ce_sl_time if ce_sl_time else end_dt
            pe_sl_time = pe_sl_time if pe_sl_time else end_dt
            min_sl_time = min(ce_sl_time, pe_sl_time)

            if (re_no < re_entries) and (min_sl_time < end_dt - datetime.timedelta(minutes=5)):
                
                start_dt = min_sl_time
                ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
                if ce_scrip is None:
                    sl_flag = False
                    exit_time = min_sl_time
                    break
                    
                trades_entries.append((ce_scrip, pe_scrip, ce_price, pe_price, start_dt))
                ce_data, pe_data = bt._get_straddle_data(start_dt, end_dt, ce_scrip, pe_scrip, seperate=True)
                common_dt = sorted(set(ce_data['date_time']) & set(pe_data['date_time']))
                scommon_dt = sorted(set(scommon_dt) & set(common_dt))
                
                ce_scrip, _, ce_price, _, ce_start_dt = max(trades_entries, key=lambda x: (x[0], x[-1]))
                _, pe_scrip, _, pe_price, pe_start_dt = min(trades_entries, key=lambda x: (x[1], x[-1]))
                
                ce_data = bt._get_single_leg_data(ce_start_dt, end_dt, ce_scrip)
                ce_data = ce_data[ce_data['date_time'].isin(scommon_dt)]
                
                pe_data = bt._get_single_leg_data(pe_start_dt, end_dt, pe_scrip)
                pe_data = pe_data[pe_data['date_time'].isin(scommon_dt)]
                                
                _, _, _, _, ce_sl_time, _ = bt.sl_check_by_given_data(ce_data, o=ce_price, sl=sl, orderside=orderside, from_candle_close=from_candle_close)
                _, _, _, _, pe_sl_time, _ = bt.sl_check_by_given_data(pe_data, o=pe_price, sl=sl, orderside=orderside, from_candle_close=from_candle_close)
            else:
                exit_time = min_sl_time

                if exit_time == end_dt:
                    sl_flag = False
                else:
                    sl_flag = True
                
        trade_logs = []
        total_pnl = 0
        for re in range(max_re+1):
            
            if len(trades_entries) > re:
                ce_scrip, pe_scrip, ce_price, pe_price, start_dt = trades_entries[re]
                _, _, _, _, _, _, pnl = bt.sl_check_combine_leg(entry_time, exit_time, ce_scrip, pe_scrip)
                total_pnl += pnl
                trade_logs.extend((start_dt, f"({ce_scrip}, {pe_scrip})", ce_price + pe_price, sl_flag, exit_time, pnl))
            else:
                trade_logs.extend(('', '', '', False, '', 0))
            
        trade_logs.append(total_pnl)
               
        return [code, bt.index, start_time, end_time, orderside, method, sl, om, re_entries, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price] + trade_logs
    
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, method, sl, om, re_entries])
        return

In [ ]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)
            max_re = 7

            log_cols = 'P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_Method/P_SL/P_OM/P_ReEntries/Date/Day/DTE/EntryTime/Future'

            for r in range(max_re+1):
                log_cols += f'/STD{r}.Start.Time/STD{r}.Scrip/STD{r}.Open/STD{r}.SL.Flag/STD{r}.Exit.Time/STD{r}.PNL'    
            log_cols += '/Total.PNL'
            log_cols = log_cols.split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [CSRE_HCLP(bt, row['entry_time'], row['exit_time'], row['orderside'], row['method'], row['sl'], row['om'], row['re_entries']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))